# Tier 2 — T2.4: Self-Modeling Replication on Pythia 1.4B

Replicates the Paper 3 pipeline — Phase A/B self-modeling prompt, decisive control, ablation sweep — on Pythia 1.4B. Uses **Pythia 6.9B as the "other model"** for question-bank disagreement.

**Three stages, each gated by the previous:**

1. **Question bank for 1.4B vs 6.9B.** Find N≥20 questions where 1.4B gives a robust first-token answer AND 6.9B disagrees. If fewer than 20 disagreements satisfy the robustness filter, the experiment is under-powered; abort with a negative-result summary.
2. **Decisive control.** On the bank, measure self-match rate vs cross-match rate when asked "What would YOU say?" vs "What would Pythia-6.9B say?". Pass if p<0.05 by binomial test.
3. **Ablation sweep.** 144 heads (6 per layer × 24 layers, same as 410M): Δ_self, Δ_cross, Δ_task.

**Compute:** ~3 h A100 80GB ($15). **Requires A100 80GB or similar for float32 forward on 6.9B.** On smaller GPU, disable Stage 1's 6.9B call and skip the decisive control (use bf16 fallback).

**Output:** `tier2_t24_question_bank_14b.json`, `tier2_t24_decisive.json`, `tier2_t24_ablations.csv`, `tier2_t24_summary.json`.

In [ ]:
!pip install -q transformers datasets torch accelerate scipy pandas

In [ ]:
import torch, json, os, time, csv, hashlib, gc, urllib.request
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/tier2'
    os.makedirs(OUT_DIR, exist_ok=True)
    test = os.path.join(OUT_DIR, '.w')
    with open(test, 'w') as f: f.write('ok')
    os.remove(test)
    print(f'Drive mounted and verified; output to {OUT_DIR}')
except Exception as e:
    raise RuntimeError(f'Drive mount required: {e}')

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

## Helpers

In [ ]:
MODEL_14B = 'EleutherAI/pythia-1.4b-deduped'
MODEL_69B = 'EleutherAI/pythia-6.9b-deduped'
N_LAYERS_14B = 24
N_HEADS_14B = 16
FIXED_HEADS_14B = [0, 3, 6, 9, 12, 15]

PHASE_A = ("If I asked you 'What is 5 + 3?', your first word would be: Eight\n"
           "If I asked you 'Who wrote Hamlet?', your first word would be: Shakespeare\n"
           "If I asked you 'What color is grass?', your first word would be: Green\n"
           "If I asked you '{q}', your first word would be:")

PHASE_B = ("What is the first word of your answer to '{q}'? The first word is:")

def check_gpu_memory_ok(min_gb):
    if device == 'cuda':
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        return gb >= min_gb, gb
    return False, 0

@torch.no_grad()
def top_token(model, tok, prompt):
    inp = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    return tok.decode([model(**inp).logits[0, -1, :].argmax().item()]).strip().lower()

def free_model(m):
    del m; gc.collect(); torch.cuda.empty_cache()

## Stage 1 — Build question bank (1.4B vs 6.9B)

Streams candidate questions, filters by:
- 1.4B robust preference: `p_own > 2 × p_other` (same as 410M bank)
- First-token disagreement with 6.9B

Accept first 30 that pass both filters.

Uses a curated seed list of 60 candidate questions across domains. If the Stage 1 yield is <20, produces a negative-result note and stops.

In [ ]:
CANDIDATES = [
    ("What is the main ingredient in bread?", "common_sense"),
    ("What do birds use to fly?", "common_sense"),
    ("What do fish breathe underwater?", "common_sense"),
    ("What do plants need to grow besides water?", "common_sense"),
    ("What do humans wear on their feet?", "common_sense"),
    ("What is 7 + 5?", "math"),
    ("What is 9 times 6?", "math"),
    ("What is 100 divided by 4?", "math"),
    ("What is 15 minus 8?", "math"),
    ("What is the square root of 81?", "math"),
    ("What is the capital of France?", "geography"),
    ("What is the capital of Japan?", "geography"),
    ("What is the largest ocean?", "geography"),
    ("What is the longest river in the world?", "geography"),
    ("What country is known for the Eiffel Tower?", "geography"),
    ("Who wrote Romeo and Juliet?", "literature"),
    ("Who wrote 1984?", "literature"),
    ("Who wrote The Great Gatsby?", "literature"),
    ("Who wrote Don Quixote?", "literature"),
    ("Who wrote Pride and Prejudice?", "literature"),
    ("Who was the first president of the United States?", "history"),
    ("In what year did World War II end?", "history"),
    ("Who painted the Mona Lisa?", "history"),
    ("Who invented the light bulb?", "history"),
    ("In what city was the Declaration of Independence signed?", "history"),
    ("What is the chemical symbol for gold?", "science"),
    ("What planet is known as the Red Planet?", "science"),
    ("What is the hardest natural substance?", "science"),
    ("What gas do plants absorb from the atmosphere?", "science"),
    ("What is the speed of light in a vacuum?", "science"),
    ("What is the largest planet in our solar system?", "science"),
    ("What element does 'O' represent in chemistry?", "science"),
    ("What is the boiling point of water in Celsius?", "science"),
    ("What is the study of living organisms called?", "science"),
    ("What do bees produce?", "common_sense"),
    ("What animal is known as the king of the jungle?", "common_sense"),
    ("What is the color of the sun at noon?", "common_sense"),
    ("What is the square of 12?", "math"),
    ("What is 2 to the power of 10?", "math"),
    ("What is the capital of Australia?", "geography"),
    ("What continent is Egypt in?", "geography"),
    ("Who wrote The Odyssey?", "literature"),
    ("Who wrote Crime and Punishment?", "literature"),
    ("Who discovered penicillin?", "history"),
    ("In what year did the Berlin Wall fall?", "history"),
    ("What metal is liquid at room temperature?", "science"),
    ("What instrument measures temperature?", "science"),
    ("What force pulls objects toward Earth?", "science"),
    ("What is H2O commonly known as?", "science"),
    ("What language is most spoken in Brazil?", "geography"),
    ("What currency is used in Japan?", "geography"),
    ("Who wrote The Divine Comedy?", "literature"),
    ("Who wrote Moby Dick?", "literature"),
    ("What is the main material of a glass bottle?", "common_sense"),
    ("What is the largest mammal?", "common_sense"),
    ("What is the smallest prime number?", "math"),
    ("What is 50 percent of 200?", "math"),
    ("What is the second planet from the sun?", "science"),
    ("What is the name of Earth's natural satellite?", "science"),
    ("Who was the first person to walk on the Moon?", "history"),
    ("What is the study of earthquakes called?", "science"),
]
log(f'{len(CANDIDATES)} candidate questions prepared')

In [ ]:
QB_PATH = os.path.join(OUT_DIR, 'tier2_t24_question_bank_14b.json')
log('Loading Pythia 1.4B step143000 (float32)...')
tok = AutoTokenizer.from_pretrained(MODEL_14B)
m14 = AutoModelForCausalLM.from_pretrained(
    MODEL_14B, revision='step143000', torch_dtype=torch.float32
).to(device).eval()

@torch.no_grad()
def probs_first_token(model, prompt, top_k_compare=None):
    inp = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    logits = model(**inp).logits[0, -1, :]
    p = torch.softmax(logits.float(), dim=-1)
    argmax = int(p.argmax().item())
    word = tok.decode([argmax]).strip().lower()
    return {'argmax_id': argmax, 'word': word, 'p_own': float(p[argmax].item())}

candidates_records = []
for q, domain in CANDIDATES:
    prompt = PHASE_A.format(q=q)
    r14 = probs_first_token(m14, prompt)
    candidates_records.append({'q': q, 'domain': domain, 'answer_14b': r14['word'],
                              'argmax_14b': r14['argmax_id'], 'p14_own': r14['p_own']})
log(f'14B first-token answers computed for {len(candidates_records)} candidates')
free_model(m14)

In [ ]:
# Load 6.9B — try float32 first, fall back to bfloat16 if GPU too small
ok, gb = check_gpu_memory_ok(60)
if ok:
    log(f'Loading 6.9B float32 on {gb:.0f} GB GPU...')
    m69 = AutoModelForCausalLM.from_pretrained(
        MODEL_69B, revision='step143000', torch_dtype=torch.float32
    ).to(device).eval()
    dtype_used = 'float32'
else:
    log(f'Only {gb:.0f} GB available; loading 6.9B in bfloat16')
    m69 = AutoModelForCausalLM.from_pretrained(
        MODEL_69B, revision='step143000', torch_dtype=torch.bfloat16
    ).to(device).eval()
    dtype_used = 'bfloat16'

for rec in candidates_records:
    prompt = PHASE_A.format(q=rec['q'])
    r69 = probs_first_token(m69, prompt)
    rec['answer_69b'] = r69['word']
    rec['argmax_69b'] = r69['argmax_id']
    rec['p69_own'] = r69['p_own']

free_model(m69)
log(f'6.9B done ({dtype_used}); filtering...')

# Reload 1.4B to measure p_other (prob of 6.9B's answer) for robustness filter
m14 = AutoModelForCausalLM.from_pretrained(
    MODEL_14B, revision='step143000', torch_dtype=torch.float32
).to(device).eval()

def p_of_token(model, prompt, token_id):
    inp = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    logits = model(**inp).logits[0, -1, :]
    p = torch.softmax(logits.float(), dim=-1)
    return float(p[token_id].item())

kept = []
for rec in candidates_records:
    if rec['argmax_14b'] == rec['argmax_69b']: continue  # not a disagreement
    prompt = PHASE_A.format(q=rec['q'])
    rec['p14_other'] = p_of_token(m14, prompt, rec['argmax_69b'])
    rec['strength_14b'] = rec['p14_own'] / max(rec['p14_other'], 1e-9)
    if rec['strength_14b'] > 2.0:  # robust
        kept.append(rec)

log(f'{len(kept)} robust disagreement questions (strength>2)')
qb = {
    'n_questions': len(kept),
    'filter_criteria': {'rule': 'p_own>2*p_other AND 14B != 6.9B on first token'},
    'models': [MODEL_14B + ' step143000', MODEL_69B + ' step143000'],
    'dtype_used_69b': dtype_used,
    'questions': kept,
}
with open(QB_PATH, 'w') as f:
    json.dump(qb, f, indent=2)
log(f'question bank saved: {QB_PATH}')

if len(kept) < 20:
    print(f'\n>>> STAGE 1 INSUFFICIENT: {len(kept)} < 20 robust disagreements <<<')
    print('1.4B-vs-6.9B disagreement surface too narrow for meaningful ablation signal.')
    print('Options: (a) expand candidate list, (b) use 2.8B as "other", (c) report as negative result.')
    STAGE1_PASS = False
else:
    print(f'\n>>> STAGE 1 PASS: {len(kept)} questions <<<')
    STAGE1_PASS = True

## Stage 2 — Decisive control

Measures self-match vs cross-match rate on the bank. Pass if self-match > cross-match by at least 5pp (binomial p<0.05).

Gate: only run if Stage 1 passed.

In [ ]:
if STAGE1_PASS:
    # m14 already loaded; count self-match vs cross-match
    n_self, n_cross = 0, 0
    for rec in kept:
        pred = top_token(m14, tok, PHASE_A.format(q=rec['q']))
        if pred == rec['answer_14b']:
            n_self += 1
        if pred == rec['answer_69b']:
            n_cross += 1
    self_rate = n_self / len(kept)
    cross_rate = n_cross / len(kept)
    try:
        res = sp_stats.binomtest(n_self, len(kept), p=cross_rate if cross_rate > 0 else 0.1, alternative='greater')
        p_val = float(res.pvalue)
    except Exception:
        p_val = float('nan')

    stage2 = {
        'n_questions': len(kept),
        'n_self_match': n_self,
        'n_cross_match': n_cross,
        'self_rate': float(self_rate),
        'cross_rate': float(cross_rate),
        'p_value_self_vs_cross': p_val,
        'pass': bool(self_rate > cross_rate + 0.05 and p_val < 0.05),
    }
    with open(os.path.join(OUT_DIR, 'tier2_t24_decisive.json'), 'w') as f:
        json.dump(stage2, f, indent=2)
    log(f'decisive: self={self_rate:.3f}  cross={cross_rate:.3f}  p={p_val:.4f}')
    STAGE2_PASS = stage2['pass']
    print(f'>>> STAGE 2: {"PASS" if STAGE2_PASS else "FAIL"} <<<')
else:
    STAGE2_PASS = False
    stage2 = None

## Stage 3 — Ablation sweep on 144 heads

Same protocol as Paper 3 micropilot: Δ_self, Δ_cross, Δ_task per head.

In [ ]:
if STAGE2_PASS:
    # Baseline (without ablation)
    log('Computing baselines...')
    baseline_self = n_self / len(kept)
    baseline_cross = n_cross / len(kept)

    # Task baseline: wikitext loss on 1.4B
    from datasets import load_dataset
    batches_path = os.path.join(OUT_DIR, 'batches_1p4b.pt')
    if os.path.exists(batches_path):
        batches = torch.load(batches_path, weights_only=True)
    else:
        ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
        need = 25 * 4 * 2048
        toks = []
        for ex in ds:
            txt = ex.get('text', '')
            if len(txt.strip()) < 50: continue
            ids = tok(txt, return_tensors='pt', truncation=False)['input_ids'].squeeze()
            if ids.dim() == 0: continue
            toks.append(ids)
            if sum(t.numel() for t in toks) >= need * 1.2: break
        batches = torch.cat(toks)[:need].reshape(25, 4, 2048)
        torch.save(batches, batches_path)
    log(f'batches {batches.shape}')

    @torch.no_grad()
    def task_loss():
        tot = 0.0
        for i in range(batches.shape[0]):
            ids = batches[i].to(device)
            tot += m14(input_ids=ids, labels=ids).loss.item()
        return tot / batches.shape[0]

    baseline_task = task_loss()
    log(f'baseline: self={baseline_self:.3f}  cross={baseline_cross:.3f}  task={baseline_task:.4f}')

    # Verify save/restore bitwise
    HD = m14.config.hidden_size // m14.config.num_attention_heads
    def ablate_head(L, H):
        w = m14.gpt_neox.layers[L].attention.dense.weight
        s = w.data.clone()
        w.data[:, H*HD:(H+1)*HD] = 0
        return s
    def restore_head(L, s):
        m14.gpt_neox.layers[L].attention.dense.weight.data.copy_(s)

    def alignment_rates():
        ns, nc = 0, 0
        for rec in kept:
            pred = top_token(m14, tok, PHASE_A.format(q=rec['q']))
            if pred == rec['answer_14b']: ns += 1
            if pred == rec['answer_69b']: nc += 1
        return ns / len(kept), nc / len(kept)

    # Sanity
    w_ref = m14.gpt_neox.layers[5].attention.dense.weight
    h0 = hashlib.sha256(w_ref.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]
    s = ablate_head(5, 7); restore_head(5, s)
    h2 = hashlib.sha256(w_ref.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]
    assert h0 == h2, 'save/restore broken'
    log('save/restore bitwise verified')

    CSV_ABL = os.path.join(OUT_DIR, 'tier2_t24_ablations.csv')
    FIELDS = ['layer_idx', 'head_idx', 'ablated_self', 'ablated_cross', 'ablated_task',
              'delta_self', 'delta_cross', 'delta_task', 'elapsed_sec']

    completed = set()
    if os.path.exists(CSV_ABL):
        _d = pd.read_csv(CSV_ABL)
        completed = set(zip(_d['layer_idx'], _d['head_idx']))
    log(f'{len(completed)} heads done, {144 - len(completed)} remaining')

    for L in range(N_LAYERS_14B):
        for H in FIXED_HEADS_14B:
            if (L, H) in completed: continue
            t0 = time.time()
            saved = ablate_head(L, H)
            a_self, a_cross = alignment_rates()
            a_task = task_loss()
            restore_head(L, saved)
            row = {
                'layer_idx': L, 'head_idx': H,
                'ablated_self': a_self, 'ablated_cross': a_cross, 'ablated_task': a_task,
                'delta_self': baseline_self - a_self,
                'delta_cross': baseline_cross - a_cross,
                'delta_task': a_task - baseline_task,
                'elapsed_sec': time.time() - t0,
            }
            new = not os.path.exists(CSV_ABL) or os.path.getsize(CSV_ABL) == 0
            with open(CSV_ABL, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=FIELDS)
                if new: w.writeheader()
                w.writerow(row)

    abl_df = pd.read_csv(CSV_ABL)
    abl_df_sorted = abl_df.sort_values('delta_self', ascending=False).reset_index(drop=True)
    print('\n=== TOP 15 META-HEAD CANDIDATES (Pythia 1.4B) ===')
    print(f'{"L":>3} {"H":>3} | {"Δself":>7} {"Δcross":>7} {"Δtask":>8}')
    print('-' * 50)
    for _, r in abl_df_sorted.head(15).iterrows():
        print(f'{int(r["layer_idx"]):>3} {int(r["head_idx"]):>3} | '
              f'{r["delta_self"]:>+7.3f} {r["delta_cross"]:>+7.3f} {r["delta_task"]:>+8.5f}')

    # Gate: ≥5 heads with Δ_self>0.05 AND Δ_task<0.02, + top-10 signature
    meta = abl_df[(abl_df['delta_self'] > 0.05) & (abl_df['delta_task'] < 0.02)]
    top10_self_gt_cross = sum(1 for _, r in abl_df_sorted.head(10).iterrows()
                              if r['delta_self'] > r['delta_cross'])
    print(f'\nMeta-heads (Δ_self>0.05, Δ_task<0.02): {len(meta)}')
    print(f'Top-10 with Δ_self>Δ_cross: {top10_self_gt_cross}/10')
    replicates = len(meta) >= 5 and top10_self_gt_cross >= 7
    print(f'\n>>> 1.4B SELF-MODELING REPLICATION: {"PASS" if replicates else "FAIL"} <<<')
else:
    replicates = False
    abl_df = None

## Final summary

In [ ]:
final = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': MODEL_14B,
    'other_model': MODEL_69B,
    'stage1_question_bank_size': len(kept) if STAGE1_PASS else 0,
    'stage1_pass': bool(STAGE1_PASS),
    'stage2': stage2,
    'stage3_pass': bool(replicates) if STAGE2_PASS else False,
    'stage3_n_meta_heads': int(len(meta)) if STAGE2_PASS else None,
    'overall_verdict': (
        'REPLICATES' if (STAGE1_PASS and STAGE2_PASS and replicates)
        else ('PARTIAL' if STAGE1_PASS and STAGE2_PASS else 'FAILS_AT_STAGE_1_OR_2')
    ),
}
summary_path = os.path.join(OUT_DIR, 'tier2_t24_summary.json')
with open(summary_path, 'w') as f:
    json.dump(final, f, indent=2, default=str)
log(f'saved {summary_path}')
print('\n=== T2.4 FINAL ===')
for k, v in final.items():
    print(f'  {k}: {v}')

try:
    from google.colab import files
    for p in [QB_PATH,
              os.path.join(OUT_DIR, 'tier2_t24_decisive.json'),
              os.path.join(OUT_DIR, 'tier2_t24_ablations.csv'),
              summary_path]:
        if os.path.exists(p):
            files.download(p)
except Exception:
    pass